# **cv2.dnn.readNetFrom() kullanarak YOLOv3**

####**Bu derste, önceden eğitilmiş bir YOLOV3 Modelini nasıl yükleyeceğimizi ve birkaç görüntü üzerinde çıkarımlar yapmak için OpenCV'yi nasıl kullanacağımızı öğreneceğiz**


In [ ]:
import numpy as np
import time
import cv2
import os
from os import listdir
from os.path import isfile, join
from matplotlib import pyplot as plt 

def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()


## **YOLO Object Detection**

![](https://opencv-tutorial.readthedocs.io/en/latest/_images/yolo1_net.png)

**Gerekli Adımlar**

1. Önceden eğitilmiş YOLOV3 ağırlıklarını kullanın
2. Yüklenen modelimiz olan blob nesnemizi oluşturun
3. Modeli çalıştıran arka ucu ayarlayın

In [ ]:
# YOLO modelimizin üzerinde eğitildiği COCO sınıf etiketlerini yükleyin
# YOLO modeli bir görüntüde nesne tespit ettiğinde sana sadece sınıfın ID numarasını döner (örneğin: 0, 1, 2 …).
# coco.names dosyasında COCO datasetindeki sınıfların isimleri sırayla yazar (örn: person, bicycle, car, …).
# Bu dosyayı okuyup LABELS listesine yüklediğinde, ID → isim eşleşmesi yapabilirsiniz.
labelsPath = "../files/YOLO/yolo/coco.names"
LABELS = open(labelsPath).read().strip().split("\n")
 
# Şimdi olası sınıf etiketini temsil etmek için bir renk listesi başlatmamız gerekiyor
# Böylece nesne tespit edildiğinde kutucuk çizilirken her sınıfa rasgele farklı bir renk verilir.
COLORS = np.random.randint(0, 255, size=(len(LABELS), 3), dtype="uint8")

print("Loading YOLO weights...") 

weights_path = "../files/YOLO/yolo/yolov3.weights"  # YOLO’nun daha önce eğitimden öğrendiği ağırlıklar 
cfg_path = "../files/YOLO/yolo/yolov3.cfg" 
# Hangi katmanlar var, her katmanda kaç filtre var, kernel boyutu, stride, aktivasyon fonksiyonları vs. bu dosyada tanımlıdır.


net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
#Artık net, eğitilmiş YOLOv3 modelini temsil eder ve görüntüler üzerinde nesne tespiti yapmaya hazırdır

# Backend ayarlayın
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
# net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
# cv2.dnn.DNN_TARGET_CPU → Modeli CPU üzerinde çalıştır
# cv2.dnn.DNN_BACKEND_OPENCV → Hesaplamaları tamamen OpenCV’nin kendi implementasyonu ile yap.
# cv2.dnn.DNN_BACKEND_CUDA → NVIDIA CUDA kullanarak GPU hızlandırmalı çalıştır.
# cv2.dnn.DNN_BACKEND_HALIDE → Halide optimizasyon framework’ünü kullan.
# cv2.dnn.DNN_BACKEND_INFERENCE_ENGINE → Intel’in OpenVINO kütüphanesi ile çalıştır.

print("YOLO Katmanları")
ln = net.getLayerNames() #Modeldeki tüm katmanların isimlerini bir liste olarak döndürür.

# There are 254 Layers
print(len(ln), ln)

### Ağın girdisi blob nesnesi olarak adlandırılır. 
### Neden blob gerekiyor?
### Ham resimler farklı boyut, renk düzeni, ölçek vs. içerebilir.
### YOLO (ve diğer CNN modelleri) sabit boyutlu giriş bekler (örneğin 416×416 piksel).
### Ayrıca pikselleri 0–255 aralığından 0–1 aralığına normalize etmek gerekir.
### Renk kanalları da (BGR → RGB) modele uygun sıraya çevrilir.



Fonksiyon ```cv.dnn.blobFromImage(image, scalefactor=1.0, size=(width, height), mean=(0,0,0), swapRB=True, crop=False)``` görüntüyü bir bloba dönüştürür:

```blob = cv.dnn.blobFromImage(img, 1/255.0, (416, 416), swapRB=True, crop=False)```

**Aşağıdaki parametrelere sahiptir:**

1. img - Girdi olarak verilen görüntü
2. scale - scale = 1/255 → piksel değerlerini 0–1 aralığına getirir.
3. size - Görüntüyü modele uygun boyuta yeniden boyutlandırır. burada 416x416 
4. mean - Genellikle model eğitilirken kullanılan datasetin kanallarının ortalaması çıkarılır.
5. swapRB - swapRB=True → BGR → RGB dönüşümü yapılır.
6. crop - Görüntü boyutu size ile tam uyuşmuyorsa kırpma yapılıp yapılmayacağını belirler.


In [ ]:
# ilgili klasörde bulunan resimleri al    
mypath = "../files/YOLO/images/"
file_names = [f for f in listdir(mypath) if isfile(join(mypath, f))]

# Görüntüler arasında döngü yaparak onları sınıflandırıcımızdan geçirelim
for file in file_names:
    # girdi resmimizi yükleyelim ve uzamsal boyutları alalım
    image = cv2.imread(mypath+file)
    (H, W) = image.shape[:2]
 
    # YOLO'dan yalnızca ihtiyacımız olan *çıktı* katman adlarını istiyoruz
    ln = net.getLayerNames() 
    ln = [ln[i - 1] for i in net.getUnconnectedOutLayers()]  
    # Ağa bağlı olmayan (yani çıktı katmanları) katmanların indekslerini verir

    # Şimdi blobumuzu girdi görüntümüzden oluşturuyoruz
    blob = cv2.dnn.blobFromImage(image, 1 / 255.0, (416, 416), swapRB=True, crop=False)
    
    # Girdimizi görüntü bloğumuza ayarlıyoruz
    net.setInput(blob)
    # Sonra ağ üzerinden bir ileri geçiş çalıştırıyoruz
    layerOutputs = net.forward(ln)

    # tespit edilen sınırlayıcı kutularımız, güvenirliklerimiz ve sınıflarımız için listelerimizi başlatıyoruz
    boxes = [] # Açıklama: Tespit edilen her nesnenin sınırlayıcı kutusunu (bounding box) tutar.
    confidences = [] # Her tespit için modelin güven skorunu saklar.  float, 0–1 arasında
    IDs = [] # Her tespit edilen nesnenin sınıf indeksini tutar.

    # Katman çıktılarının her biri üzerinde döngü
    for output in layerOutputs:

        # Her algılama üzerinde döngü
        for detection in output:
            # Sınıf kimliğini ve tespit olasılığını elde edelim
            scores = detection[5:]
            classID = np.argmax(scores)
            confidence = scores[classID]

            # Sadece en olası tahminleri saklıyoruz
            if confidence > 0.75:
                # Sınırlayıcı kutu koordinatlarını görüntüye göre ölçeklendiriyoruz
                # Not: YOLO aslında sınırlayıcı kutunun merkezini (x, y) 
                # ve ardından kutunun genişliğini ve yüksekliğini döndürür
                box = detection[0:4] * np.array([W, H, W, H])
                (centerX, centerY, width, height) = box.astype("int")

                # Sınırlayıcı kutunun üst ve sol köşesini alın
                # Zaten genişlik ve yüksekliğe sahip olduğumuzu unutmayın
                x = int(centerX - (width / 2))
                y = int(centerY - (height / 2))

                # Sınırlayıcı kutu koordinatları, güvenirlikler ve sınıf kimlikleri listemizi ekleyin
                boxes.append([x, y, int(width), int(height)])
                confidences.append(float(confidence))
                IDs.append(classID)

    # Şimdi üst üste binen sınırlayıcı kutuları azaltmak için maksimum olmayan bastırma uyguluyoruz
    idxs = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.3)

    # Bir nesne bulunduğunda devam ediyoruz
    if len(idxs) > 0:
        # tuttuğumuz indeksler üzerinde yineleme
        for i in idxs.flatten():
            # Sınırlayıcı kutu koordinatlarını alın
            (x, y) = (boxes[i][0], boxes[i][1])
            (w, h) = (boxes[i][2], boxes[i][3])

            # Sınırlayıcı kutularımızı çizin ve sınıf etiketimizi resmin üzerine yerleştirin
            color = [int(c) for c in COLORS[IDs[i]]]
            cv2.rectangle(image, (x, y), (x + w, y + h), color, 3)
            text = "{}: {:.4f}".format(LABELS[IDs[i]], confidences[i])
            cv2.putText(image, text, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # çıktı görüntüsünü göster
    imshow("YOLO Detections", image, size = 6)